# Reproduce the frozen text open-model extension on Colab

Choose **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU), then run these cells in order in a fresh runtime. This notebook runs the pinned **SmolLM2 1.7B** and **Granite 3.3 2B** on the original SST2 and TREC pilot datasets, with zero-shot and four labeled examples per class: **8 conditions, 1,600 predictions**, using the same 200 test rows for each condition within a dataset. TREC has six classes (24 training demonstrations at four per class); SST2 has two (8 demonstrations).

Create `artifacts/jev-text-extension-colab.zip` locally with `python scripts/export_text_extension_colab.py`, then upload that exact bundle below. If the inline upload widget is unavailable, upload the ZIP through Colab’s **Files sidebar** so it is available at `/content/jev-text-extension-colab.zip`; the same hash and inventory checks apply. Its archive, source, preset, tokenizer and data identities are checked before execution. Dataset attribution and license notes are preserved in the included manifests and `configs/datasets.json`. The frozen data retains the original preparation and character-prefix policy; it is not re-prepared here.

The new wrapper reuses the unchanged numeric extension's fixed-system, thinking-disabled chat renderer and the original numeric-label-plus-EOS likelihood scorer. It has its own source identity. Public weights are loaded with Hugging Face authentication explicitly disabled; no secrets, hosted APIs, adapters or Drive mounting are used. Prompts and label sequences must fit the 8,192-token limit without truncation. Models run sequentially in float16 on the requested CUDA device; failures do not trigger precision or device fallback.


In [ ]:
# Upload and verify the source/data bundle before extracting or executing it.
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, io, json, os, stat, tempfile, zipfile

EXPECTED_BUNDLE_SHA = "30620cce39a27521551e76a73db30228ad9b4e116d37c87e86a678183884d80d"
EXPECTED_WRAPPER_SHA = "d3f6c06f6138da02a38e49ab220ed3a184c047b3b4a4729eb01908e513334526"
EXPECTED_HELPER_SHA = "b0c07e82e03b79040f13bd7810baf92bcd3a30fe8aa3b48d0704baedea8801a3"
EXPECTED_PRESETS_SHA = "ed176a77f40fd8cd842672193333559a8221b3ac9dd52259d275c54da4cf6421"
EXPECTED_CORE_SHA = "d5547ccde4224e182653315d81c0a631c789bbe294ad7bcf95ef0cddd25ce608"
PREFIX = "jev-text-extension"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

def require(condition, message):
    if not condition:
        raise ValueError(message)

def sha(content):
    return hashlib.sha256(content).hexdigest()

# The Files sidebar is an alternative when the inline upload widget is unavailable.
uploaded_path = Path("/content/jev-text-extension-colab.zip")
uploaded = None
if uploaded_path.is_file():
    bundle = uploaded_path.read_bytes()
else:
    uploaded = files.upload()
    require(len(uploaded) == 1, "Upload exactly one source ZIP")
    bundle = next(iter(uploaded.values()))
require(sha(bundle) == EXPECTED_BUNDLE_SHA, "Source ZIP differs from the pinned bundle; stop and inspect")
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    inventory = archive.infolist()
    names = [item.filename for item in inventory]
    require(len(names) == len(set(names)), "Duplicate archive member")
    require(sum(item.file_size for item in inventory) <= 20_000_000, "Unexpected source archive size")
    for item in inventory:
        name = PurePosixPath(item.filename)
        require(not name.is_absolute() and ".." not in name.parts and "\\" not in item.filename,
                "Unsafe source archive path")
        require(name.parts[0] == PREFIX and not item.is_dir() and not stat.S_ISLNK(item.external_attr >> 16),
                "Unexpected source archive member")
    manifest = json.loads(archive.read(f"{PREFIX}/bundle_manifest.json"))
    expected_names = {f"{PREFIX}/{name}" for name in manifest["files_sha256"]}
    require(set(names) == expected_names | {f"{PREFIX}/bundle_manifest.json"}, "Bundle inventory differs")
    require(manifest["schema_version"] == 1, "Unexpected bundle schema")
    require(manifest["core_sha256"] == EXPECTED_CORE_SHA, "Frozen core pin differs")
    require(manifest["text_wrapper_sha256"] == EXPECTED_WRAPPER_SHA
            == manifest["files_sha256"]["scripts/run_text_extension_local.py"], "Text wrapper pin differs")
    require(manifest["numeric_helper_sha256"] == EXPECTED_HELPER_SHA, "Renderer helper pin differs")
    require(manifest["preset_file_sha256"] == EXPECTED_PRESETS_SHA, "Preset pin differs")
    require(manifest["files_sha256"]["scripts/run_expanded_numeric_local.py"] == EXPECTED_HELPER_SHA,
            "Helper source pin differs")
    require(manifest["files_sha256"]["configs/expanded_numeric_models.json"] == EXPECTED_PRESETS_SHA,
            "Model preset pin differs")
    verified = {}
    for name, expected in manifest["files_sha256"].items():
        content = archive.read(f"{PREFIX}/{name}")
        require(sha(content) == expected, f"Source member hash differs: {name}")
        verified[name] = content
    scratch = Path(tempfile.mkdtemp(prefix="jev-text-extension-", dir="/content"))
    PROJECT = scratch / PREFIX
    PROJECT.mkdir()
    for name, content in verified.items():
        destination = PROJECT / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)
    (PROJECT / "bundle_manifest.json").write_text(json.dumps(manifest, indent=2))
del uploaded, bundle, verified
os.chdir(PROJECT)
print("Verified project:", PROJECT)
print("Source members:", len(manifest["files_sha256"]), "| Frozen core:", EXPECTED_CORE_SHA)


In [ ]:
# Install the pinned inference stack without changing the source or presets.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev,neural]",
                "transformers==4.57.6", "huggingface-hub==0.36.2"], cwd=PROJECT, check=True)

# Some Colab images bundle an incompatible optional torchao build. This
# float16 experiment does not use torchao or quantization. Repair only that case.
smoke_code = ("from transformers.models.llama.modeling_llama import LlamaForCausalLM; "
              "from transformers.models.granite.modeling_granite import GraniteForCausalLM")
smoke = subprocess.run([sys.executable, "-c", smoke_code], cwd=PROJECT, capture_output=True, text=True)
torchao_removed = False
if smoke.returncode and "torchao" in smoke.stderr:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
    torchao_removed = True
    smoke = subprocess.run([sys.executable, "-c", smoke_code], cwd=PROJECT, capture_output=True, text=True)
if smoke.returncode:
    print(smoke.stderr)
    smoke.check_returncode()
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_text_extension_local.py"],
               cwd=PROJECT, check=True)


In [ ]:
# Confirm CUDA and record the actual runtime before running all eight conditions.
import importlib.metadata, platform
import torch
require(torch.cuda.is_available(), "CUDA is unavailable; select a GPU runtime. No CPU fallback is allowed.")
require(importlib.metadata.version("transformers") == "4.57.6", "Transformers version differs")
require(importlib.metadata.version("huggingface-hub") == "0.36.2", "Hub version differs")
COMMAND = [sys.executable, "scripts/run_text_extension_local.py", "--device", "cuda",
           "--model-keys", "smollm2", "granite", "--shots", "0", "4",
           "--datasets", "sst2", "trec", "--allow-download", "--execute"]
packages = {}
for package in ("torch", "transformers", "huggingface-hub", "accelerate", "peft", "numpy", "scipy", "scikit-learn"):
    try:
        packages[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        pass
runtime = {"schema_version": 1, "source_bundle_sha256": EXPECTED_BUNDLE_SHA,
           "wrapper_sha256": EXPECTED_WRAPPER_SHA, "helper_sha256": EXPECTED_HELPER_SHA, "preset_sha256": EXPECTED_PRESETS_SHA,
           "core_sha256": EXPECTED_CORE_SHA, "python": platform.python_version(),
           "packages": packages, "device": "cuda", "dtype": "float16",
           "gpu_name": torch.cuda.get_device_name(0),
           "gpu_total_memory_bytes": torch.cuda.get_device_properties(0).total_memory,
           "torchao_removed_after_import_failure": torchao_removed,
           "command": COMMAND[1:], "expected_conditions": 8, "expected_predictions": 1600,
           "hosted_calls": 0, "adapter_training": False}
local = PROJECT / "results/text_extension/local"
(local / "plans").mkdir(parents=True, exist_ok=True)
(local / "plans/colab-runtime-preflight.json").write_text(json.dumps(runtime, indent=2) + "\n")
print(json.dumps(runtime, indent=2))
with subprocess.Popen(COMMAND, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1) as inference:
    for line in inference.stdout:
        print(line, end="", flush=True)
    require(inference.wait() == 0, "Inference stopped; preserve checkpoints and inspect the output above")


In [ ]:
# Re-audit all eight completed sources and export only checksummed local evidence.
sys.path[:0] = [str(PROJECT / "src"), str(PROJECT / "scripts")]
from export_text_extension_colab import export_results
archive_path = export_results(PROJECT)
print("Archive SHA256:", sha(archive_path.read_bytes()))
files.download(str(archive_path))


After downloading, verify the displayed SHA256 and import from the local repository:

```bash
.venv/bin/python scripts/import_text_extension_colab.py /path/to/jev-text-extension-results.zip --expected-sha256 REPLACE_WITH_PRINTED_ARCHIVE_SHA256 --execute
```

Omit `--execute` to audit the downloaded archive without importing. The importer verifies the producer/data pins, exact member inventory, per-file checksums, complete condition coverage, rendered prompt identities and measured metrics. It refuses to replace differing evidence. Import receipts retain archive, manifest and member hashes. The result ZIP contains only local prediction/run/test evidence and preflight/runtime metadata.

The inference cell can resume saved row checkpoints within the same runtime and project directory. Do not rerun the upload cell when resuming: it creates a fresh directory. To preserve a fully completed model before the other finishes, run `python scripts/export_text_extension_colab.py --results --model-keys smollm2` (or `granite`) in the project directory and download its ZIP; import with the same `--model-keys` selection. Partial conditions are refused.

An out-of-memory error stops execution. Use a sufficiently large CUDA runtime while keeping the same precision, prompts and configuration. Cross-hardware numerical results need not be bit-identical; actual package, GPU and runtime metadata is retained. These small public datasets support exploratory comparisons; their possible model-training exposure and the fixed label-plus-EOS protocol limit generalization.
